In [1]:
#Defining the storage paths
storage_account_name = "nyctaxi100"
container_name = "taxidata"

raw_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/raw/"
processed_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/processed/"
curated_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/curated/"

print("Paths have been configured")

StatementMeta(taxisparkpool, 6, 2, Finished, Available, Finished, False)

Paths have been configured


In [2]:
from pyspark.sql.functions import col, hour, dayofweek, month, unix_timestamp
import pyspark.sql.functions as F
from pyspark.sql.types import *

# Disable vectorized reading
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")
spark.conf.set("spark.sql.legacy.parquet.int96RebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.legacy.parquet.datetimeRebaseModeInRead", "CORRECTED")

# List of all 12 months
months = [
    "2023-01", "2023-02", "2023-03", "2023-04",
    "2023-05", "2023-06", "2023-07", "2023-08",
    "2023-09", "2023-10", "2023-11", "2023-12"
]

failed_months = []
total_rows = 0

for m in months:
    try:
        print(f"Processing {m}...")
        
        # Read single file
        file_path = f"{raw_path}yellow_tripdata_{m}.parquet"
        df = spark.read \
            .option("inferSchema", "false") \
            .parquet(file_path)
        
        # Cast all columns to correct types
        df = df.select(
            col("VendorID").cast("long"),
            col("tpep_pickup_datetime").cast("timestamp"),
            col("tpep_dropoff_datetime").cast("timestamp"),
            col("passenger_count").cast("double"),
            col("trip_distance").cast("double"),
            col("RatecodeID").cast("double"),
            col("store_and_fwd_flag").cast("string"),
            col("PULocationID").cast("long"),
            col("DOLocationID").cast("long"),
            col("payment_type").cast("long"),
            col("fare_amount").cast("double"),
            col("extra").cast("double"),
            col("mta_tax").cast("double"),
            col("tip_amount").cast("double"),
            col("tolls_amount").cast("double"),
            col("improvement_surcharge").cast("double"),
            col("total_amount").cast("double"),
            col("congestion_surcharge").cast("double"),
            col("airport_fee").cast("double")
        )
        
        # Clean data
        df = df.dropna(subset=[
            "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
            "passenger_count", "trip_distance", "fare_amount", "total_amount"
        ])
        
        df = df.filter(
            (col("passenger_count") > 0) & (col("passenger_count") <= 6) &
            (col("trip_distance") > 0) & (col("trip_distance") <= 100) &
            (col("fare_amount") > 0) & (col("fare_amount") <= 500) &
            (col("total_amount") > 0)
        )
        
        df = df.dropDuplicates()

        # Add analytics columns
        df = df \
            .withColumn("pickup_hour", F.hour(col("tpep_pickup_datetime"))) \
            .withColumn("pickup_dayofweek", F.dayofweek(col("tpep_pickup_datetime"))) \
            .withColumn("pickup_month", F.month(col("tpep_pickup_datetime"))) \
            .withColumn("trip_duration_mins",
                F.round((F.unix_timestamp("tpep_dropoff_datetime") -
                       F.unix_timestamp("tpep_pickup_datetime")) / 60, 2)) \
            .withColumn("tip_percentage",
                F.round((col("tip_amount") / col("fare_amount")) * 100, 2))

        #Counting the number of rows
        row_count = df.count()
        total_rows += row_count

        # Save to curated
        output_path = f"{curated_path}yellow_tripdata_{m}.parquet"
        df.write.mode("overwrite").parquet(output_path)
        
        print(f"✅ {m} done! Rows: {row_count:,}")
        
    except Exception as e:
        print(f"❌ {m} failed: {str(e)[:200]}")
        failed_months.append(m)

print(f"Total rows saved: {total_rows:,}")
print(f"Failed months: {failed_months}")
print("All done!")

StatementMeta(taxisparkpool, 6, 3, Finished, Available, Finished, False)

Processing 2023-01...
✅ 2023-01 done! Rows: 2,884,169
Processing 2023-02...
✅ 2023-02 done! Rows: 2,732,598
Processing 2023-03...
✅ 2023-03 done! Rows: 3,190,569
Processing 2023-04...
✅ 2023-04 done! Rows: 3,077,711
Processing 2023-05...
✅ 2023-05 done! Rows: 3,281,584
Processing 2023-06...
✅ 2023-06 done! Rows: 3,084,643
Processing 2023-07...
✅ 2023-07 done! Rows: 2,712,812
Processing 2023-08...
✅ 2023-08 done! Rows: 2,628,728
Processing 2023-09...
✅ 2023-09 done! Rows: 2,604,671
Processing 2023-10...
✅ 2023-10 done! Rows: 3,243,631
Processing 2023-11...
✅ 2023-11 done! Rows: 3,092,553
Processing 2023-12...
✅ 2023-12 done! Rows: 3,076,230

Total rows saved: 35,609,899
Failed months: []
All done!
